# Prompting Notebook

This notebook loads precomputed RAG results from `rag_outputs/all_queries_results.json`, builds prompts containing top chunks per policy, and (after you add your API key) queries GPT-4.1 mini for answers.



In [19]:
import os
from openai import OpenAI

In [20]:
import json
from pathlib import Path
from typing import Dict, Any, List

RESULTS_PATH = Path("rag_outputs/all_queries_results.json")
assert RESULTS_PATH.exists(), f"File not found: {RESULTS_PATH}"

with RESULTS_PATH.open("r", encoding="utf-8") as f:
    ALL_RESULTS: Dict[str, Any] = json.load(f)

available_queries = list(ALL_RESULTS.keys())
print(f"✅ Loaded {len(available_queries)} queries from {RESULTS_PATH}")


✅ Loaded 1 queries from rag_outputs/all_queries_results.json


In [21]:
from dataclasses import dataclass

@dataclass
class Chunk:
    policy_name: str
    text: str
    score: float


def get_top_chunks(query: str, policies_to_use: int = 2, chunks_per_policy: int = 3) -> List[Chunk]:
    record = ALL_RESULTS[query]
    policies = record["policies"]
    chunks: List[Chunk] = []
    for policy_id in sorted(policies.keys())[:policies_to_use]:
        policy_data = policies[policy_id]
        policy_name = policy_data["policy_name"]
        for item in policy_data["results"][:chunks_per_policy]:
            chunks.append(
                Chunk(
                    policy_name=policy_name,
                    text=item.get("embedding_text", item.get("text", "")),
                    score=item.get("reranked_score") or item.get("hybrid_score", 0.0),
                )
            )
    return chunks

print("✅ Chunk extraction helper ready.")


✅ Chunk extraction helper ready.


In [22]:
PROMPT_TEMPLATE = """
You are an insurance policy assistant. Use only the provided chunks to answer the user question.
If the chunks do not contain enough information, clearly state what is missing.
Mention policy names when referencing specific details.

User Question:
{question}

Context:
{context}

Answer Guidelines:
- Ground every statement in the context above.
- Compare policies when applicable. 
- Provide a score ranging from 1 to 5 for each policy when comparing policy based on the User Question. Return the score in the following format: <Policy_Name>Your answer regarding to the S
core in the format of integer ranging from 1-5</Policy_Name>. The score must be a integer ranging from 1-5, and provide the rationale before you output the score.
- Use concise sentences and bullet points if they improve clarity.
"""


def build_prompt(question: str, chunks: List[Chunk]) -> str:
    context_lines = []
    for idx, chunk in enumerate(chunks, 1):
        context_lines.append(
            f"[{idx}] ({chunk.policy_name}) score={chunk.score:.3f}\n{chunk.text}"
        )
    context_text = "\n\n".join(context_lines)
    return PROMPT_TEMPLATE.format(question=question, context=context_text)

print("✅ Prompt template ready.")


✅ Prompt template ready.


In [ ]:
def call_llm(prompt: str) -> str:
    """Placeholder for GPT-4.1 mini call. Replace with actual API invocation."""
#    Example once you add your API key:
    
    # replace your open ai key in the line below
    my_open_ai_key =''
    client = OpenAI(api_key=my_open_ai_key)

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content
    return "[LLM placeholder] Replace with GPT-4.1 mini API call."


def answer_query(query_key: str, policies_to_use: int = 2, chunks_per_policy: int = 3, show_prompt: bool = False) -> Dict[str, Any]:
    assert query_key in ALL_RESULTS, f"Unknown query: {query_key}"
    question = ALL_RESULTS[query_key]["query"]
    chunks = get_top_chunks(query_key, policies_to_use=policies_to_use, chunks_per_policy=chunks_per_policy)
    prompt = build_prompt(question, chunks)
    if show_prompt:
        print("======= PROMPT SENT TO LLM =======")
        print(prompt)
        print("======= END PROMPT =======")
    answer = call_llm(prompt)
    return {
        "question": question,
        "answer": answer,
        "chunks": chunks,
        "prompt": prompt,
    }


def run_demo(query_key: str, **kwargs):
    print(f"Available queries: {available_queries}")
    result = answer_query(query_key, **kwargs)
    print("\nQUESTION:\n", result["question"])
    print("\nANSWER:\n", result["answer"])
    print("\nCONTEXT CHUNKS:\n")
    for idx, chunk in enumerate(result["chunks"], 1):
        print(f"[{idx}] ({chunk.policy_name}) score={chunk.score:.3f}\n{chunk.text}\n{'-'*80}")
    return result

# Example:
# run_demo("what is the deductible for each policy", show_prompt=True)



In [15]:
run_demo("deductible comparison", show_prompt=True)

Available queries: ['deductible comparison', 'emergency room coverage comparison', 'compare coverage for pregnancy and diabetes between policies', 'copmare copay for primary doctor visit']
======= PROMPT SENT TO LLM =======

You are an insurance policy assistant. Use only the provided chunks to answer the user question.
If the chunks do not contain enough information, clearly state what is missing.
Mention policy names when referencing specific details.

User Question:
deductible comparison

Context:
[1] (policy1.pdf) score=0.752
When using a Participating Provider3 Calendar Year medical Deductible Individual coverage $0 Family coverage $0: individual $0: Family Calendar Year pharmacy Deductible Individual coverage Family coverage $0 $0: individual $0: Family Calendar Year Out-of-Pocket Maximum4 An Out-of-Pocket Maximum is the most a Member will pay for Covered Services each Calendar Year. Any exceptions are listed in the EOC. Keywords: provider, $0, out-of-pocket, deductible, coverage

{'question': 'deductible comparison',
 'answer': 'Here is a deductible comparison based on the provided information:\n\n**Policy 1 (policy1.pdf):**  \n- Calendar Year medical Deductible: $0 for Individual and Family coverage.  \n- Calendar Year pharmacy Deductible: $0 for Individual and Family coverage.  \n- This means no deductible must be met before the plan pays for covered services.  \n- Some services have copayments (e.g., $30 per office visit), but no deductible applies.  \n- Out-of-Pocket Maximum is mentioned but no deductible cost to meet.  \n\n**Policy 2 (policy2.pdf):**  \n- Overall deductible is $7,500 Individual and $15,000 Family.  \n- Each family member must meet their own individual deductible until the family deductible is met.  \n- Some services (preventive, copayment services, certain prescriptions) are covered before meeting the deductible.  \n- No separate deductibles for specific services.  \n- Deductible must be met before the plan pays for most covered services. 

In [7]:
run_demo("emergency room coverage comparison", show_prompt=True)

Available queries: ['what is the deductible for each policy', 'emergency room coverage comparison', 'prescription drug costs', 'out of pocket maximum']
======= PROMPT SENT TO LLM =======

You are an insurance policy assistant. Use only the provided chunks to answer the user question.
If the chunks do not contain enough information, clearly state what is missing.
Mention policy names when referencing specific details.

User Question:
emergency room coverage comparison

Context:
[1] (policy1.pdf) score=1.517
$0 Tubal ligation $0 Vasectomy $0 Medical nutrition therapy, not related to diabetes $0 Pregnancy and maternity care Physician office visits: prenatal and initial postnatal $0 Abortion and abortion-related services $0 3 Benefits5 Your payment When using a Participating Provider3 CYD2 applies Emergency Services Emergency room services $250visit If admitted to the Hospital, this payment for emergency room services does not apply. Keywords: provider, $0, emergency, benefit, $250 Costs: 

{'question': 'emergency room coverage comparison',
 'answer': 'Here is a comparison of emergency room coverage based on the provided policy information:\n\n**Policy 1:**\n- Emergency room services require a $250 copay per visit.\n- If admitted to the hospital, the ER copay does not apply (covered under inpatient services).\n- Physician services in the emergency room are $0 copay.\n- Ambulance services cost $150 per transport.\n- Urgent care center visits cost $30 per visit.\n- Overall, the ER visit copay is fixed and relatively moderate.\n\n**Policy 2:**\n- Emergency room care requires 50% coinsurance.\n- No specific copay amount is given; costs depend on total charges.\n- Ambulance and emergency medical transportation also have 50% coinsurance.\n- Urgent care visits cost $75 per visit, deductible does not apply.\n- Hospital facility fees (including ER if admitted) have 50% coinsurance.\n- Deductibles are high ($7,500 in examples), which may increase out-of-pocket costs before coverage

In [16]:
run_demo("copmare copay for primary doctor visit", show_prompt=True)

Available queries: ['deductible comparison', 'emergency room coverage comparison', 'compare coverage for pregnancy and diabetes between policies', 'copmare copay for primary doctor visit']
======= PROMPT SENT TO LLM =======

You are an insurance policy assistant. Use only the provided chunks to answer the user question.
If the chunks do not contain enough information, clearly state what is missing.
Mention policy names when referencing specific details.

User Question:
copmare copay for primary doctor visit

Context:
[1] (policy1.pdf) score=1.606
For example, you may owe an office visit payment in addition to an allergy serum payment when you visit the doctor for an allergy shot. 6 Preventive Health Services: If you only receive Preventive Health Services during a Physician office visit, there is no Copayment or Coinsurance for the visit. If you receive both Preventive Health Services and other Covered Services during the Physician office visit, you may have a Copayment or Coinsurance 

{'question': 'copmare copay for primary doctor visit',
 'answer': 'Here is a comparison of the copay for a primary care doctor visit based on the provided policy information:\n\n**Policy1 (Blue Shield of California):**  \n- Primary care office visit copay is $30 per visit.  \n- Preventive health services have no copay if only preventive services are received; otherwise, a copay or coinsurance may apply.  \n- Participating providers must be used for primary care.  \n- Clear, fixed copay amount for primary care visits.  \n\n**Policy2:**  \n- Primary care visit copay is $50 per visit; deductible does not apply.  \n- Preventive services have no cost sharing.  \n- Specialist copay is higher ($100), but for primary care, it is $50.  \n- Virtual visits are available.  \n\n**Comparison and Scores:**  \n- Policy1 offers a lower copay ($30) for primary care visits compared to Policy2 ($50).  \n- Both waive copays for preventive services only visits.  \n- Policy1 provides more detailed cost struc

In [24]:
run_demo("coverage for pregnancy and diabetes", show_prompt=True)

Available queries: ['coverage for pregnancy and diabetes']
======= PROMPT SENT TO LLM =======

You are an insurance policy assistant. Use only the provided chunks to answer the user question.
If the chunks do not contain enough information, clearly state what is missing.
Mention policy names when referencing specific details.

User Question:
coverage for pregnancy and diabetes

Context:
[1] (policy1.pdf) score=1.579
$0 Tubal ligation $0 Vasectomy $0 Medical nutrition therapy, not related to diabetes $0 Pregnancy and maternity care Physician office visits: prenatal and initial postnatal $0 Abortion and abortion-related services $0 3 Benefits5 Your payment When using a Participating Provider3 CYD2 applies Emergency Services Emergency room services $250visit If admitted to the Hospital, this payment for emergency room services does not apply. Keywords: provider, $0, emergency, benefit, $250 Costs: $0, $250

[2] (policy1.pdf) score=0.887
Other services and supplies Diabetes care services D

{'question': 'coverage for pregnancy and diabetes',
 'answer': 'Coverage for pregnancy and diabetes based on the provided chunks:\n\n**Pregnancy Coverage:**\n\n- **Policy1:**\n  - Pregnancy and maternity care including physician office visits for prenatal and initial postnatal care are covered with $0 cost to the insured. (Chunk [1])\n  - Abortion and abortion-related services are covered at $0 cost. (Chunk [1])\n  - No specific mention of copayments or deductibles for pregnancy visits, implying minimal or no cost sharing for these services.\n\n- **Policy2:**\n  - Office visits for pregnant women have a copayment: $50 for primary care initial visit and $100 for specialist initial visit; deductible does not apply. (Chunk [5])\n  - Copayment applies only to the first prenatal visit per pregnancy.\n  - Cost sharing does not apply for preventive services.\n  - Abortion coverage is limited except when pregnancy endangers the woman’s life. (Chunk [4])\n\n**Diabetes Coverage:**\n\n- **Policy1

In [27]:
call_llm('1+1?')

'1 + 1 = 2'